#Set up

In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# Função do gráfico

In [ ]:
def pareto_plotly(
    df_raw,
    col_categoria="aluno_profissional_escolaridade_descricao",
    col_id="aluno_id",
    metodo_contagem="nunique",
    titulo="Número de Alunos por Escolaridade — Pareto",
    titulo_eixo_x="Número de Alunos",
    titulo_eixo_y=None,
    titulo_eixo_superior="Acumulado (%)",
    corescale_barras="Blues",
    cor_curva="black",
    espessura_curva=3,
    suavizar_curva=False,
    exibir_texto_barras=True,
    exibir_texto_curva=True,
    exibir_colorbar=True,
    titulo_colorbar="Número de Alunos",
    th_A=80,
    th_B=95,
    mostrar_faixas_abc=True,
    cor_faixa_A="rgba(46, 204, 113, 0.18)",
    cor_faixa_B="rgba(243, 156, 18, 0.18)",
    cor_faixa_C="rgba(231, 76, 60, 0.16)",
    mostrar_linhas_corte=True,
    cor_linhas_corte="gray",
    dash_linhas_corte="dash",
    mostrar_rotulos_abc=True,
    categoria_destacar="Não identificado",
    cor_destacada="crimson",
    largura_px=1200,
    altura_px=800,
    margem_esquerda=300,
    margem_direita=140,
    margem_superior=100,
    margem_inferior=100,
    dominio_x=(0.0, 0.80),
    multiplicador_folga_x=1.45,
    fonte_titulo=22,
    fonte_subtitulo=16,
    fonte_eixo=18,
    fonte_ticks=14,
    fonte_categorias=16,
    fonte_rotulos_barras=14,
    fonte_rotulos_curva=12,
    fonte_abc=22,
    mostrar_legenda=True,
    legenda_y=-0.18,
    standoff_titulo_eixo_x=20,
    standoff_titulo_eixo_y=40,
    standoff_titulo_eixo_superior=10,
):
    """
    Cria um gráfico de Pareto horizontal com curva acumulada e classificação ABC.

    Estrutura do gráfico:
    1. Agrega os dados por categoria
    2. Ordena do maior para o menor
    3. Calcula percentual individual e acumulado
    4. Plota barras horizontais
    5. Plota curva acumulada no eixo superior
    6. Adiciona áreas ABC e linhas de corte
    7. Permite destacar uma categoria específica

    Parâmetros principais
    ---------------------
    df_raw : pandas.DataFrame
        Base original, ainda não agregada.

    col_categoria : str
        Coluna categórica usada no eixo Y.

    col_id : str | None
        Coluna identificadora usada quando o método de contagem for "nunique".

    metodo_contagem : str
        "nunique" para contar IDs únicos.
        "size" para contar linhas.

    titulo : str
        Título principal do gráfico.

    titulo_eixo_x : str
        Título do eixo X principal.

    titulo_eixo_y : str | None
        Título do eixo Y.
        Se None, usa o nome da coluna em col_categoria.

    titulo_eixo_superior : str
        Título do eixo superior, da curva acumulada.

    standoff_titulo_eixo_x, standoff_titulo_eixo_y, standoff_titulo_eixo_superior : int
        Controlam a distância dos títulos dos eixos para seus respectivos eixos.
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    # ==========================================================
    # 1. FUNÇÕES AUXILIARES INTERNAS
    # ==========================================================
    def _validar_inputs():
        if col_categoria not in df_raw.columns:
            raise ValueError(f"Coluna de categoria '{col_categoria}' não encontrada no DataFrame.")

        if col_id is not None and col_id not in df_raw.columns:
            raise ValueError(f"Coluna de ID '{col_id}' não encontrada no DataFrame.")

    def _agregar_dados():
        """
        Constrói a base agregada do Pareto.

        Regras:
        - nunique: conta IDs únicos por categoria
        - size: conta linhas por categoria
        """
        metodo_efetivo = metodo_contagem

        if col_id is None and metodo_efetivo == "nunique":
            metodo_efetivo = "size"

        if metodo_efetivo == "nunique" and col_id is not None:
            grp = df_raw.groupby(col_categoria, dropna=False)[col_id].nunique()
        else:
            grp = df_raw.groupby(col_categoria, dropna=False).size()

        df = grp.rename("count").reset_index()

        # Ordenação clássica do Pareto
        df = df.sort_values("count", ascending=False).reset_index(drop=True)

        total = df["count"].sum()
        df["pct"] = df["count"] / total * 100.0
        df["cum_pct"] = df["pct"].cumsum()

        # Texto das barras
        df["label_texto"] = df.apply(
            lambda r: f"{int(r['count']):,}".replace(",", ".") + f" ({r['pct']:.1f}%)",
            axis=1
        )

        return df

    def _definir_cores_barras(df):
        """
        Mantém degradê padrão, mas permite cor fixa para categoria destacada.
        """
        if categoria_destacar is None:
            return df["count"].tolist()

        cores = []
        for categoria, valor in zip(df[col_categoria], df["count"]):
            if categoria == categoria_destacar:
                cores.append(cor_destacada)
            else:
                cores.append(valor)
        return cores

    def _adicionar_barras(fig, df):
        """
        Adiciona as barras horizontais do Pareto.
        """
        marker_dict = dict(
            color=_definir_cores_barras(df),
            colorscale=corescale_barras,
        )

        if exibir_colorbar:
            marker_dict["colorbar"] = dict(
                title=titulo_colorbar,
                x=1.12,
                xanchor="left"
            )
            marker_dict["showscale"] = True
        else:
            marker_dict["showscale"] = False

        fig.add_trace(
            go.Bar(
                y=df[col_categoria],
                x=df["count"],
                orientation="h",
                marker=marker_dict,
                text=df["label_texto"] if exibir_texto_barras else None,
                textposition="outside",
                textfont=dict(size=fonte_rotulos_barras),
                name="Contagem",
            )
        )

    def _adicionar_curva(fig, df):
        """
        Adiciona a curva acumulada no eixo superior x2.
        """
        line_shape = "spline" if suavizar_curva else "linear"

        fig.add_trace(
            go.Scatter(
                x=df["cum_pct"],
                y=df[col_categoria],
                mode="lines+markers+text" if exibir_texto_curva else "lines+markers",
                line=dict(color=cor_curva, width=espessura_curva, shape=line_shape),
                marker=dict(
                    size=8,
                    color="white",
                    line=dict(color=cor_curva, width=2)
                ),
                text=[f"{v:.1f}%" for v in df["cum_pct"]] if exibir_texto_curva else None,
                textposition="middle right",
                textfont=dict(size=fonte_rotulos_curva),
                name="Acumulado (%)",
                xaxis="x2",
            )
        )

    def _adicionar_linhas_corte(fig):
        if not mostrar_linhas_corte:
            return

        for th in [th_A, th_B]:
            fig.add_shape(
                type="line",
                xref="x2",
                yref="paper",
                x0=th,
                x1=th,
                y0=0,
                y1=1,
                line=dict(color=cor_linhas_corte, width=2, dash=dash_linhas_corte),
            )

    def _adicionar_faixas_abc(fig):
        if not mostrar_faixas_abc:
            return

        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=0, x1=th_A, y0=0, y1=1,
            fillcolor=cor_faixa_A,
            line=dict(width=0),
            layer="below",
        )

        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=th_A, x1=th_B, y0=0, y1=1,
            fillcolor=cor_faixa_B,
            line=dict(width=0),
            layer="below",
        )

        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=th_B, x1=100, y0=0, y1=1,
            fillcolor=cor_faixa_C,
            line=dict(width=0),
            layer="below",
        )

    def _adicionar_rotulos_abc(fig):
        if not mostrar_rotulos_abc:
            return

        fig.add_annotation(
            x=th_A / 2, y=0.5,
            xref="x2", yref="paper",
            text="<b>A</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

        fig.add_annotation(
            x=(th_A + th_B) / 2, y=0.5,
            xref="x2", yref="paper",
            text="<b>B</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

        fig.add_annotation(
            x=(th_B + 100) / 2, y=0.5,
            xref="x2", yref="paper",
            text="<b>C</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

    def _configurar_layout(fig, df):
        xmax = df["count"].max() * multiplicador_folga_x

        texto_titulo_eixo_y = titulo_eixo_y if titulo_eixo_y is not None else col_categoria

        fig.update_layout(
            title=dict(
                text=titulo,
                x=0.5,
                y=0.97,
                font=dict(size=fonte_titulo, family="Arial Black", color="#1f2c56")
            ),
            width=largura_px,
            height=altura_px,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(
                l=margem_esquerda,
                r=margem_direita,
                t=margem_superior,
                b=margem_inferior
            ),

            # Eixo X principal - contagem
            xaxis=dict(
                domain=list(dominio_x),
                title=dict(
                    text=titulo_eixo_x,
                    standoff=standoff_titulo_eixo_x,
                    font=dict(size=fonte_eixo)
                ),
                range=[0, xmax],
                showgrid=True,
                gridcolor="rgba(0,0,0,0.08)",
                tickfont=dict(size=fonte_ticks),
            ),

            # Eixo X superior - acumulado
            xaxis2=dict(
                overlaying="x",
                side="top",
                domain=list(dominio_x),
                range=[0, 100],
                tick0=0,
                dtick=20,
                ticksuffix="%",
                showgrid=False,
                title=dict(
                    text=titulo_eixo_superior,
                    standoff=standoff_titulo_eixo_superior,
                    font=dict(size=fonte_subtitulo)
                ),
                tickfont=dict(size=fonte_ticks - 2)
            ),

            # Eixo Y com categorias
            yaxis=dict(
                autorange="reversed",
                title=dict(
                    text=texto_titulo_eixo_y,
                    standoff=standoff_titulo_eixo_y,
                    font=dict(size=fonte_eixo)
                ),
                tickfont=dict(size=fonte_categorias, color="black"),
                automargin=True
            ),

            legend=dict(
                orientation="h",
                y=legenda_y,
                x=0.5,
                xanchor="center"
            ) if mostrar_legenda else None,

            showlegend=mostrar_legenda
        )

        fig.update_traces(selector=dict(type="scatter"), cliponaxis=False)

    # ==========================================================
    # 2. PIPELINE PRINCIPAL
    # ==========================================================
    _validar_inputs()
    df_ordenado = _agregar_dados()

    fig = go.Figure()

    _adicionar_barras(fig, df_ordenado)
    _adicionar_curva(fig, df_ordenado)
    _adicionar_linhas_corte(fig)
    _adicionar_faixas_abc(fig)
    _adicionar_rotulos_abc(fig)
    _configurar_layout(fig, df_ordenado)

    return fig

In [ ]:
def pareto_plotly(
    df_raw,
    col_categoria="aluno_profissional_escolaridade_descricao",
    col_id="aluno_id",
    metodo_contagem="nunique",
    titulo="Número de Alunos por Escolaridade — Pareto",
    titulo_eixo_x="Número de Alunos",
    titulo_eixo_y=None,
    titulo_eixo_superior="Acumulado (%)",
    corescale_barras="Blues",
    cor_curva="black",
    espessura_curva=3,
    suavizar_curva=False,
    exibir_texto_barras=True,
    exibir_texto_curva=True,
    exibir_colorbar=True,
    titulo_colorbar="Número de Alunos",
    th_A=80,
    th_B=95,
    mostrar_faixas_abc=True,
    cor_faixa_A="rgba(46, 204, 113, 0.18)",
    cor_faixa_B="rgba(243, 156, 18, 0.18)",
    cor_faixa_C="rgba(231, 76, 60, 0.16)",
    mostrar_linhas_corte=True,
    cor_linhas_corte="gray",
    dash_linhas_corte="dash",
    mostrar_rotulos_abc=True,
    categoria_destacar="Não identificado",
    cor_destacada="crimson",
    largura_px=1200,
    altura_px=800,
    margem_esquerda=300,
    margem_direita=140,
    margem_superior=100,
    margem_inferior=100,
    dominio_x=(0.0, 0.80),
    multiplicador_folga_x=1.45,
    fonte_titulo=22,
    fonte_subtitulo=16,
    fonte_eixo=18,
    fonte_ticks=14,
    fonte_categorias=16,
    fonte_rotulos_barras=14,
    fonte_rotulos_curva=12,
    fonte_abc=22,
    mostrar_legenda=True,
    legenda_y=-0.18,
    standoff_titulo_eixo_x=20,
    standoff_titulo_eixo_y=40,
    standoff_titulo_eixo_superior=10,
):
    """
    Cria um gráfico de Pareto horizontal com curva acumulada e classificação ABC.

    Estrutura do gráfico:
    1. Agrega os dados por categoria
    2. Ordena do maior para o menor
    3. Calcula percentual individual e acumulado
    4. Plota barras horizontais
    5. Plota curva acumulada no eixo superior
    6. Adiciona áreas ABC e linhas de corte
    7. Permite destacar uma categoria específica

    Parâmetros principais
    ---------------------
    df_raw : pandas.DataFrame
        Base original, ainda não agregada.

    col_categoria : str
        Coluna categórica usada no eixo Y.

    col_id : str | None
        Coluna identificadora usada quando o método de contagem for "nunique".

    metodo_contagem : str
        "nunique" para contar IDs únicos.
        "size" para contar linhas.

    titulo : str
        Título principal do gráfico.

    titulo_eixo_x : str
        Título do eixo X principal.

    titulo_eixo_y : str | None
        Título do eixo Y.
        Se None, usa o nome da coluna em col_categoria.

    titulo_eixo_superior : str
        Título do eixo superior, da curva acumulada.

    corescale_barras : str
        Escala de cores para as barras (ex: 'Blues', 'Viridis').

    cor_curva : str
        Cor da linha da curva acumulada.

    espessura_curva : int
        Espessura da linha da curva acumulada.

    suavizar_curva : bool
        Se True, a curva acumulada será suavizada (spline). Se False, será linear.

    exibir_texto_barras : bool
        Se True, exibe o texto com contagem e percentual nas barras.

    exibir_texto_curva : bool
        Se True, exibe o texto com percentual na curva acumulada.

    exibir_colorbar : bool
        Se True, exibe a barra de cores (colorbar) para as barras.

    titulo_colorbar : str
        Título da barra de cores.

    th_A : int
        Percentual limite para a faixa A (ex: 80 para os primeiros 80%).

    th_B : int
        Percentual limite para a faixa B (ex: 95 para os primeiros 95%).

    mostrar_faixas_abc : bool
        Se True, exibe as faixas de cores para a classificação ABC.

    cor_faixa_A, cor_faixa_B, cor_faixa_C : str
        Cores para as faixas A, B e C, respectivamente, em formato RGBA.

    mostrar_linhas_corte : bool
        Se True, exibe as linhas de corte verticais nos limites das faixas ABC.

    cor_linhas_corte : str
        Cor das linhas de corte.

    dash_linhas_corte : str
        Estilo de traço das linhas de corte (ex: 'dash', 'dot').

    mostrar_rotulos_abc : bool
        Se True, exibe os rótulos 'A', 'B', 'C' nas respectivas faixas.

    categoria_destacar : str
        Nome de uma categoria específica para destacar com uma cor diferente.

    cor_destacada : str
        Cor para a categoria destacada.

    largura_px : int
        Largura total do gráfico em pixels.

    altura_px : int
        Altura total do gráfico em pixels.

    margem_esquerda, margem_direita, margem_superior, margem_inferior : int
        Margens do gráfico em pixels.

    dominio_x : tuple
        Tupla (start, end) definindo a proporção do layout ocupada pelo eixo X principal.

    multiplicador_folga_x : float
        Multiplicador para definir o limite máximo do eixo X, adicionando uma folga.

    fonte_titulo, fonte_subtitulo, fonte_eixo, fonte_ticks, fonte_categorias, fonte_rotulos_barras, fonte_rotulos_curva, fonte_abc : int
        Tamanhos de fonte para os diversos elementos de texto no gráfico.

    mostrar_legenda : bool
        Se True, exibe a legenda do gráfico.

    legenda_y : float
        Posição vertical da legenda (em coordenadas normalizadas, onde 0 é a parte inferior e 1 é a superior).

    standoff_titulo_eixo_x, standoff_titulo_eixo_y, standoff_titulo_eixo_superior : int
        Controlam a distância dos títulos dos eixos para seus respectivos eixos.
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    # ==========================================================
    # 1. FUNÇÕES AUXILIARES INTERNAS
    # ==========================================================
    # Valida se as colunas especificadas (col_categoria, col_id) existem no DataFrame de entrada.
    def _validar_inputs():
        if col_categoria not in df_raw.columns:
            raise ValueError(f"Coluna de categoria '{col_categoria}' não encontrada no DataFrame.")

        if col_id is not None and col_id not in df_raw.columns:
            raise ValueError(f"Coluna de ID '{col_id}' não encontrada no DataFrame.")

    # Agrega os dados brutos de acordo com o método de contagem e calcula os percentuais.
    def _agregar_dados():
        """
        Constrói a base agregada do Pareto.

        Regras:
        - nunique: conta IDs únicos por categoria
        - size: conta linhas por categoria
        """
        metodo_efetivo = metodo_contagem

        # Se col_id não é fornecido e o método é 'nunique', muda para 'size' para evitar erro.
        if col_id is None and metodo_efetivo == "nunique":
            metodo_efetivo = "size"

        # Realiza a agregação dos dados.
        if metodo_efetivo == "nunique" and col_id is not None:
            grp = df_raw.groupby(col_categoria, dropna=False)[col_id].nunique()
        else:
            grp = df_raw.groupby(col_categoria, dropna=False).size()

        df = grp.rename("count").reset_index()

        # Ordena os dados em ordem decrescente pela contagem, que é o princípio do Pareto.
        df = df.sort_values("count", ascending=False).reset_index(drop=True)

        # Calcula o total para os percentuais.
        total = df["count"].sum()
        # Calcula o percentual individual de cada categoria.
        df["pct"] = df["count"] / total * 100.0
        # Calcula o percentual acumulado, essencial para a curva de Pareto e classificação ABC.
        df["cum_pct"] = df["pct"].cumsum()

        # Formata o texto que será exibido nas barras (contagem e percentual).
        df["label_texto"] = df.apply(
            lambda r: f"{int(r['count']):,}".replace(",", ".") + f" ({r['pct']:.1f}%)",
            axis=1
        )

        return df

    # Define as cores para as barras, permitindo destacar uma categoria específica.
    def _definir_cores_barras(df):
        """
        Mantém degradê padrão, mas permite cor fixa para categoria destacada.
        """
        # Se nenhuma categoria for especificada para destaque, retorna os valores de contagem para o colorscale.
        if categoria_destacar is None:
            return df["count"].tolist()

        cores = []
        # Itera sobre as categorias para atribuir cores.
        for categoria, valor in zip(df[col_categoria], df["count"]):
            # Se a categoria atual for a de destaque, usa a cor_destacada.
            if categoria == categoria_destacar:
                cores.append(cor_destacada)
            # Caso contrário, usa o valor da contagem para o colorscale padrão.
            else:
                cores.append(valor)
        return cores

    # Adiciona as barras horizontais ao objeto Figure do Plotly.
    def _adicionar_barras(fig, df):
        """
        Adiciona as barras horizontais do Pareto.
        """
        # Configurações do marcador das barras, incluindo a cor e a escala de cores.
        marker_dict = dict(
            color=_definir_cores_barras(df),
            colorscale=corescale_barras,
        )

        # Adiciona o colorbar se exibido.
        if exibir_colorbar:
            marker_dict["colorbar"] = dict(
                title=titulo_colorbar,
                x=1.12,
                xanchor="left"
            )
            marker_dict["showscale"] = True
        else:
            marker_dict["showscale"] = False

        # Adiciona o trace de barras horizontais ao gráfico.
        fig.add_trace(
            go.Bar(
                y=df[col_categoria],
                x=df["count"],
                orientation="h",
                marker=marker_dict,
                text=df["label_texto"] if exibir_texto_barras else None, # Texto exibido nas barras.
                textposition="outside",
                textfont=dict(size=fonte_rotulos_barras),
                name="Contagem", # Nome para a legenda.
            )
        )

    # Adiciona a curva acumulada de percentual ao objeto Figure do Plotly, usando um eixo X secundário.
    def _adicionar_curva(fig, df):
        """
        Adiciona a curva acumulada no eixo superior x2.
        """
        # Define a forma da linha da curva (linear ou suavizada).
        line_shape = "spline" if suavizar_curva else "linear"

        # Adiciona o trace de linha da curva acumulada.
        fig.add_trace(
            go.Scatter(
                x=df["cum_pct"],
                y=df[col_categoria],
                mode="lines+markers+text" if exibir_texto_curva else "lines+markers", # Modo de exibição (linha, marcadores, texto).
                line=dict(color=cor_curva, width=espessura_curva, shape=line_shape),
                marker=dict(
                    size=8,
                    color="white",
                    line=dict(color=cor_curva, width=2)
                ),
                text=[f"{v:.1f}%" for v in df["cum_pct"]] if exibir_texto_curva else None, # Texto para a curva.
                textposition="middle right",
                textfont=dict(size=fonte_rotulos_curva),
                name="Acumulado (%)", # Nome para a legenda.
                xaxis="x2", # Associa este trace ao eixo x2 (eixo superior).
            )
        )

    # Adiciona linhas de corte verticais no gráfico para indicar os limites das faixas ABC.
    def _adicionar_linhas_corte(fig):
        if not mostrar_linhas_corte:
            return

        # Adiciona linhas de corte para os thresholds A e B.
        for th in [th_A, th_B]:
            fig.add_shape(
                type="line",
                xref="x2", # Linha de corte referente ao eixo x2 (acumulado).
                yref="paper",
                x0=th,
                x1=th,
                y0=0,
                y1=1,
                line=dict(color=cor_linhas_corte, width=2, dash=dash_linhas_corte), # Estilo da linha.
            )

    # Adiciona as faixas de cores para a classificação ABC no fundo do gráfico.
    def _adicionar_faixas_abc(fig):
        if not mostrar_faixas_abc:
            return

        # Faixa A
        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=0, x1=th_A, y0=0, y1=1,
            fillcolor=cor_faixa_A,
            line=dict(width=0), # Sem borda para o retângulo.
            layer="below", # Para que fique atrás das barras e da curva.
        )

        # Faixa B
        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=th_A, x1=th_B, y0=0, y1=1,
            fillcolor=cor_faixa_B,
            line=dict(width=0),
            layer="below",
        )

        # Faixa C
        fig.add_shape(
            type="rect",
            xref="x2", yref="paper",
            x0=th_B, x1=100, y0=0, y1=1,
            fillcolor=cor_faixa_C,
            line=dict(width=0),
            layer="below",
        )

    # Adiciona os rótulos 'A', 'B', 'C' às respectivas faixas de classificação.
    def _adicionar_rotulos_abc(fig):
        if not mostrar_rotulos_abc:
            return

        # Rótulo para a faixa A
        fig.add_annotation(
            x=th_A / 2, y=0.5, # Posição do rótulo (centro da faixa A).
            xref="x2", yref="paper",
            text="<b>A</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

        # Rótulo para a faixa B
        fig.add_annotation(
            x=(th_A + th_B) / 2, y=0.5, # Posição do rótulo (centro da faixa B).
            xref="x2", yref="paper",
            text="<b>B</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

        # Rótulo para a faixa C
        fig.add_annotation(
            x=(th_B + 100) / 2, y=0.5, # Posição do rótulo (centro da faixa C).
            xref="x2", yref="paper",
            text="<b>C</b>",
            showarrow=False,
            font=dict(size=fonte_abc, color="rgba(0,0,0,0.6)")
        )

    # Configura o layout geral do gráfico, incluindo títulos, eixos, margens, etc.
    def _configurar_layout(fig, df):
        # Calcula o limite máximo para o eixo X principal com base na maior contagem e uma folga.
        xmax = df["count"].max() * multiplicador_folga_x

        # Define o título do eixo Y, usando o nome da coluna de categoria se não for especificado.
        texto_titulo_eixo_y = titulo_eixo_y if titulo_eixo_y is not None else col_categoria

        fig.update_layout(
            # Configura o título principal do gráfico.
            title=dict(
                text=titulo,
                x=0.5, # Centraliza o título.
                y=0.97,
                font=dict(size=fonte_titulo, family="Arial Black", color="#1f2c56")
            ),
            width=largura_px,
            height=altura_px,
            paper_bgcolor="white", # Cor de fundo do papel.
            plot_bgcolor="white", # Cor de fundo da área de plotagem.
            margin=dict(
                l=margem_esquerda,
                r=margem_direita,
                t=margem_superior,
                b=margem_inferior
            ),

            # Eixo X principal - contagem das barras.
            xaxis=dict(
                domain=list(dominio_x), # Define a porção horizontal que o eixo X ocupa.
                title=dict(
                    text=titulo_eixo_x,
                    standoff=standoff_titulo_eixo_x,
                    font=dict(size=fonte_eixo)
                ),
                range=[0, xmax],
                showgrid=True,
                gridcolor="rgba(0,0,0,0.08)",
                tickfont=dict(size=fonte_ticks),
            ),

            # Eixo X superior - acumulado (curva de Pareto).
            xaxis2=dict(
                overlaying="x", # Sobrepõe o eixo X principal.
                side="top", # Posiciona no topo do gráfico.
                domain=list(dominio_x),
                range=[0, 100],
                tick0=0,
                dtick=20,
                ticksuffix="%", # Adiciona '%' aos ticks do eixo.
                showgrid=False,
                title=dict(
                    text=titulo_eixo_superior,
                    standoff=standoff_titulo_eixo_superior,
                    font=dict(size=fonte_subtitulo)
                ),
                tickfont=dict(size=fonte_ticks - 2)
            ),

            # Eixo Y com categorias.
            yaxis=dict(
                autorange="reversed", # Inverte a ordem das categorias para que a maior fique em cima.
                title=dict(
                    text=texto_titulo_eixo_y,
                    standoff=standoff_titulo_eixo_y,
                    font=dict(size=fonte_eixo)
                ),
                tickfont=dict(size=fonte_categorias, color="black"),
                automargin=True # Ajusta automaticamente as margens para acomodar os rótulos do eixo Y.
            ),

            # Configurações da legenda.
            legend=dict(
                orientation="h", # Orientação horizontal.
                y=legenda_y,
                x=0.5,
                xanchor="center"
            ) if mostrar_legenda else None,

            showlegend=mostrar_legenda
        )

        # Garante que os textos dos traces scatter (curva) não sejam cortados pelas margens.
        fig.update_traces(selector=dict(type="scatter"), cliponaxis=False)

    # ==========================================================
    # 2. PIPELINE PRINCIPAL
    # ==========================================================
    _validar_inputs() # Primeiro, valida as entradas do usuário.
    df_ordenado = _agregar_dados() # Agrega e ordena os dados para o Pareto.

    fig = go.Figure() # Inicializa o objeto Figure do Plotly.

    _adicionar_barras(fig, df_ordenado) # Adiciona as barras ao gráfico.
    _adicionar_curva(fig, df_ordenado) # Adiciona a curva acumulada.
    _adicionar_linhas_corte(fig) # Adiciona as linhas de corte para as faixas ABC.
    _adicionar_faixas_abc(fig) # Adiciona as faixas coloridas ABC.
    _adicionar_rotulos_abc(fig) # Adiciona os rótulos 'A', 'B', 'C'.
    _configurar_layout(fig, df_ordenado) # Configura o layout final do gráfico.

    return fig # Retorna o objeto Figure configurado.

# Dados de demostração

In [ ]:
df_raw = pd.DataFrame({
    "aluno_id": np.random.randint(1, 500, 1000),
    "aluno_profissional_escolaridade_descricao": np.random.choice(
        [
            "Fundamental Completo",
            "Médio Completo",
            "Superior Completo",
            "Pós-graduação",
            "Não identificado"
        ],
        size=1000,
        p=[0.2, 0.3, 0.25, 0.2, 0.05]
    )
})

In [ ]:
df_raw.head(5)

,aluno_id,aluno_profissional_escolaridade_descricao
0,354,Superior Completo
1,351,Superior Completo
2,181,Superior Completo
3,191,Superior Completo
4,232,Superior Completo


# Plotagem

In [ ]:
fig = pareto_plotly(
    df_raw=df_raw,
    col_categoria="aluno_profissional_escolaridade_descricao",
    col_id="aluno_id",
    metodo_contagem="nunique",
    titulo="Número de Alunos por Escolaridade — Pareto",
    titulo_eixo_x="Número de Alunos",
    titulo_eixo_y="Escolaridade do aluno",
    titulo_eixo_superior="Acumulado (%)",
    categoria_destacar="Não identificado",
    cor_destacada="crimson",
    th_A=80,
    th_B=95,
    largura_px=1200,
    altura_px=800,
)

fig.show()